# Skeletize RadHAR with SK-DGCNN

This notebook applies the paper-compatible MARS SK-DGCNN checkpoint to RadHAR/MMActivity and builds `datasets/SKradHAR` with centered 2D `[x, z]` skeleton sequences. RadHAR axes are aligned to the MARS convention, target frames are translated to the MARS spatial centroid, and normalized target-domain outliers are clipped to the bound used elsewhere for RadHAR.

In [1]:
from functools import lru_cache
from pathlib import Path
import json
import shutil

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [2]:
PROJECT_ROOT = Path('/Users/elo/Coding/radar_ml')
MARS_ROOT = PROJECT_ROOT / 'datasets' / 'MARS'
RADHAR_ROOT = PROJECT_ROOT / 'datasets' / 'RadHAR'
SKRADHAR_ROOT = PROJECT_ROOT / 'datasets' / 'SKradHAR'
CHECKPOINT_PATH = SKRADHAR_ROOT / 'model-2.pt'

MARS_ACTIVITY_CLASS_NAMES = (
    'Left_upper_limb_extension', 'Right_upper_limb_extension', 'Both_upper_limb_extension',
    'Left_front_lunge', 'Right_front_lunge', 'Squad', 'Left_side_lunge',
    'Right_side_lunge', 'Left_limb_extension', 'Right_limb_extension',
)
SKELETON_JOINT_NAMES = (
    'SpineBase', 'SpineMid', 'Neck', 'Head', 'ShoulderLeft', 'ElbowLeft', 'WristLeft',
    'ShoulderRight', 'ElbowRight', 'WristRight', 'HipLeft', 'KneeLeft', 'AnkleLeft',
    'FootLeft', 'HipRight', 'KneeRight', 'AnkleRight', 'FootRight', 'SpineShoulder',
)
SKELETON_JOINT_INDICES = (0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20)
SKELETON_COORDINATE_INDICES = (0, 2)
CENTER_JOINT_INDEX = SKELETON_JOINT_NAMES.index('SpineShoulder')
RADAR_FEATURE_ROWS = {'x': 2, 'y': 3, 'z': 4, 'velocity': 5, 'intensity': 6}

DEVICE = 'mps'
CONFIG = {
    'feature_names': ('x', 'y', 'z', 'velocity', 'intensity'),
    'feature_amount': 5,
    'num_points': 25,
    'normalized_feature_clip': 4.0,
    'validation_fraction': 0.2,
    'k': 20,
    'emb_dims': 512,
    'dropout': 0.4,
    'eca_kernel_size': 3,
    'batch_size': 64,
}

# data

In [3]:
def split_mars_file_paths():
    "Split MARS training recordings for pose checkpoint validation."
    train_files, valid_files = [], []
    for class_name in MARS_ACTIVITY_CLASS_NAMES:
        paths = sorted((MARS_ROOT / 'Train').glob(f'subject*/{class_name}/radar_data*.mat'))
        valid_count = max(1, int(len(paths) * CONFIG['validation_fraction']))
        valid_count = min(valid_count, len(paths) - 1)
        train_files.extend(str(path) for path in paths[:-valid_count])
        valid_files.extend(str(path) for path in paths[-valid_count:])
    return train_files, valid_files

In [4]:
@lru_cache(maxsize=None)
def load_mars_radar_frames_raw(file_path):
    "Load raw MARS radar frames using the features expected by SK-DGCNN."
    with h5py.File(file_path, 'r') as handle:
        radar = np.asarray(handle['radar_data_cropped'])
    frame_ids = radar[0].astype(np.int64)
    rows = [RADAR_FEATURE_ROWS[name] for name in CONFIG['feature_names']]
    features = radar[rows].T.astype(np.float32)
    return [torch.from_numpy(features[frame_ids == frame_id]) for frame_id in np.unique(frame_ids)]

In [5]:
@lru_cache(maxsize=None)
def load_mars_skeletons_raw(file_path):
    "Load the synchronized 19-joint Kinect targets for one MARS radar recording."
    skeleton_path = Path(str(file_path).replace('/radar_data', '/kinect_data'))
    skeletons = []
    with h5py.File(skeleton_path, 'r') as handle:
        refs = np.asarray(handle['kinect_data_cropped']['JointPositions']).ravel()
        for ref in refs:
            joints = np.asarray(handle[ref], dtype=np.float32)[0][:, SKELETON_JOINT_INDICES].T
            skeletons.append(torch.from_numpy(joints[:, SKELETON_COORDINATE_INDICES].reshape(-1)))
    return skeletons

In [6]:
@lru_cache(maxsize=None)
def mars_radar_normalization_stats():
    "Compute SK-DGCNN radar normalization statistics from MARS training recordings."
    train_files, _ = split_mars_file_paths()
    points = torch.cat([torch.cat(load_mars_radar_frames_raw(path), dim=0) for path in train_files], dim=0)
    return points.mean(dim=0), points.std(dim=0).clamp_min(1e-6)

In [7]:
def normalize_skeleton(skeleton):
    "Center one 2D skeleton on SpineShoulder, matching pose training."
    skeleton = skeleton.view(len(SKELETON_JOINT_INDICES), len(SKELETON_COORDINATE_INDICES))
    return (skeleton - skeleton[CENTER_JOINT_INDEX]).reshape(-1)

In [8]:
@lru_cache(maxsize=None)
def load_radhar_radar_frames_raw(file_path):
    "Load RadHAR frames mapped to MARS coordinates: lateral, depth, vertical."
    tracked_keys = {'point_id', *CONFIG['feature_names']}
    points = []
    current_point = {}
    with Path(file_path).open('r') as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if line == '---':
                points.append(current_point)
                current_point = {}
            elif ':' in line:
                key, raw_value = line.split(':', 1)
                if key.strip() in tracked_keys:
                    current_point[key.strip()] = float(raw_value)
    if current_point:
        points.append(current_point)
    frames = []
    current_frame = []
    for point in points:
        if int(point.get('point_id', -1)) == 0 and current_frame:
            frames.append(torch.tensor(current_frame, dtype=torch.float32))
            current_frame = []
        current_frame.append([float(point.get('y', 0.0)), float(point.get('x', 0.0)), float(point.get('z', 0.0)), float(point.get('velocity', 0.0)), float(point.get('intensity', 0.0))])
    if current_frame:
        frames.append(torch.tensor(current_frame, dtype=torch.float32))
    return frames

In [9]:
def align_target_frame(points):
    "Translate a target-dataset radar frame to the MARS training spatial centroid."
    mars_mean, _ = mars_radar_normalization_stats()
    points = points.clone()
    points[:, :3] += mars_mean[:3] - points[:, :3].mean(dim=0)
    return points

In [10]:
def process_pose_frame(points):
    "Normalize, crop, and pad a radar frame already expressed in MARS axes."
    mean, std = mars_radar_normalization_stats()
    points = (points - mean) / std
    output = torch.zeros(CONFIG['num_points'], CONFIG['feature_amount'], dtype=torch.float32)
    output[:min(len(points), CONFIG['num_points'])] = points[:CONFIG['num_points']]
    return output

In [11]:
def process_target_frame(points):
    "Prepare aligned cross-dataset input while bounding normalized radar outliers."
    frame = process_pose_frame(align_target_frame(points))
    return frame.clamp(-CONFIG['normalized_feature_clip'], CONFIG['normalized_feature_clip'])

In [12]:
def build_mars_frame_index(file_paths):
    "Create synchronized MARS radar-to-skeleton frame indexes."
    samples = []
    for path in file_paths:
        count = min(len(load_mars_radar_frames_raw(path)), len(load_mars_skeletons_raw(path)))
        samples.extend((path, frame_idx) for frame_idx in range(count))
    return samples

In [13]:
class MARSPoseValidationDataset(Dataset):
    "MARS validation frames and centered Kinect targets."
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, frame_idx = self.samples[idx]
        target = normalize_skeleton(load_mars_skeletons_raw(path)[frame_idx])
        return process_pose_frame(load_mars_radar_frames_raw(path)[frame_idx]), target

# architecture

In [14]:
def knn(x, k):
    "Find point-neighborhood indexes for SK-DGCNN EdgeConv."
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    distance = -xx - inner - xx.transpose(2, 1)
    return distance.topk(k=k, dim=-1)[1]

In [15]:
def get_graph_feature(x, k=20):
    "Build EdgeConv features exactly as in MARS training: center minus neighbor."
    batch_size, feature_amount, point_amount = x.shape
    idx = knn(x, k=k)
    idx_base = torch.arange(batch_size, device=x.device).view(-1, 1, 1) * point_amount
    idx = (idx + idx_base).view(-1)
    points = x.transpose(2, 1).contiguous()
    neighbors = points.view(batch_size * point_amount, feature_amount)[idx]
    neighbors = neighbors.view(batch_size, point_amount, k, feature_amount)
    centers = points.view(batch_size, point_amount, 1, feature_amount).repeat(1, 1, k, 1)
    return torch.cat((centers - neighbors, centers), dim=3).permute(0, 3, 1, 2).contiguous()

In [16]:
class ECALayer(nn.Module):
    "Efficient channel attention layer used by the trained SK-DGCNN."
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, kernel_size=CONFIG['eca_kernel_size'], padding=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        weights = x.mean(dim=tuple(range(2, x.dim()))).unsqueeze(1)
        weights = self.conv(weights).squeeze(1).view(x.size(0), x.size(1), *([1] * (x.dim() - 2)))
        return x * self.sigmoid(weights)

In [17]:
class DGCNNSkeletonRegressor(nn.Module):
    "SK-DGCNN regressor matching the MARS pose checkpoint architecture."
    def __init__(self):
        super().__init__()
        self.k = CONFIG['k']
        self.joints = len(SKELETON_JOINT_INDICES)
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm1d(CONFIG['emb_dims'])
        self.bn5 = nn.BatchNorm1d(512)
        self.bn6 = nn.BatchNorm1d(self.joints * 128)
        self.conv1 = nn.Sequential(nn.Conv2d(CONFIG['feature_amount'] * 2, 64, 1, bias=False), self.bn1, nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64, 1, bias=False), self.bn2, nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), self.bn3, nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv1d(256, CONFIG['emb_dims'], 1, bias=False), self.bn4, nn.LeakyReLU(0.2))
        self.eca1 = ECALayer()
        self.eca2 = ECALayer()
        self.eca3 = ECALayer()
        self.eca4 = ECALayer()
        self.linear1 = nn.Linear(CONFIG['emb_dims'] * 2, 512, bias=False)
        self.linear2 = nn.Linear(512, self.joints * 128, bias=False)
        self.dp1 = nn.Dropout(CONFIG['dropout'])
        self.dp2 = nn.Dropout(CONFIG['dropout'])
        self.x_head = nn.Linear(128, 1)
        self.z_head = nn.Linear(128, 1)

    def forward(self, x):
        batch_size = x.size(0)
        x1 = self.eca1(self.conv1(get_graph_feature(x, self.k))).max(dim=-1)[0]
        x2 = self.eca2(self.conv2(get_graph_feature(x1, self.k))).max(dim=-1)[0]
        x3 = self.eca3(self.conv3(get_graph_feature(x2, self.k))).max(dim=-1)[0]
        x = self.eca4(self.conv4(torch.cat((x1, x2, x3), dim=1)))
        x = torch.cat((F.adaptive_max_pool1d(x, 1).view(batch_size, -1), F.adaptive_avg_pool1d(x, 1).view(batch_size, -1)), dim=1)
        x = self.dp1(F.leaky_relu(self.bn5(self.linear1(x)), negative_slope=0.2))
        x = self.dp2(F.leaky_relu(self.bn6(self.linear2(x)), negative_slope=0.2)).view(batch_size, self.joints, 128)
        return torch.cat((self.x_head(x), self.z_head(x)), dim=2).reshape(batch_size, -1)

# inference

In [18]:
def pose_joint_mae(predictions, targets):
    "Compute centered x,z coordinate MAE in centimeters for every joint."
    errors = (predictions - targets).abs().view(-1, len(SKELETON_JOINT_INDICES), len(SKELETON_COORDINATE_INDICES))
    return errors.mean(dim=(0, 2)) * 100

In [19]:
def evaluate_pose_checkpoint(checkpoint_path, file_paths):
    "Score one pose checkpoint on centered MARS x,z skeleton coordinates."
    model = DGCNNSkeletonRegressor().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
    model.eval()
    loader = DataLoader(MARSPoseValidationDataset(build_mars_frame_index(file_paths)), batch_size=CONFIG['batch_size'], shuffle=False)
    predictions = []
    targets = []
    with torch.no_grad():
        for point_cloud, target in loader:
            predictions.append(model(point_cloud.to(DEVICE).permute(0, 2, 1)).cpu())
            targets.append(target)
    joint_maes = pose_joint_mae(torch.cat(predictions), torch.cat(targets))
    return joint_maes.mean().item(), joint_maes

In [20]:
def load_pose_model(checkpoint_path):
    "Load the paper-compatible MARS SK-DGCNN checkpoint for inference."
    model = DGCNNSkeletonRegressor().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
    model.eval()
    return model

In [21]:
def normalized_radhar_feature_summary():
    "Report aligned RadHAR radar distributions in MARS-normalized space."
    sources = sorted(RADHAR_ROOT.glob('*/*/*.txt'))
    points = torch.cat([torch.cat([align_target_frame(frame) for frame in load_radhar_radar_frames_raw(str(path))], dim=0) for path in sources], dim=0)
    mean, std = mars_radar_normalization_stats()
    normalized = (points - mean) / std
    clipped = normalized.clamp(-CONFIG['normalized_feature_clip'], CONFIG['normalized_feature_clip'])
    return {
        'before_clipping': {name: {'mean': float(normalized[:, i].mean()), 'std': float(normalized[:, i].std())} for i, name in enumerate(CONFIG['feature_names'])},
        'after_clipping': {name: {'mean': float(clipped[:, i].mean()), 'std': float(clipped[:, i].std())} for i, name in enumerate(CONFIG['feature_names'])},
    }

In [22]:
def preprocess_radhar_file(source_path, destination_path, model):
    "Predict and save normalized x,z skeleton frames for one RadHAR recording."
    frames = load_radhar_radar_frames_raw(str(source_path))
    predictions = []
    with torch.no_grad():
        for start in range(0, len(frames), CONFIG['batch_size']):
            batch = torch.stack([process_target_frame(frame) for frame in frames[start:start + CONFIG['batch_size']]])
            predictions.append(model(batch.to(DEVICE).permute(0, 2, 1)).cpu())
    skeletons = torch.cat(predictions).view(-1, len(SKELETON_JOINT_INDICES), 2)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(skeletons, destination_path)

In [23]:
def prepare_skradhar(checkpoint_path, force=False):
    "Rebuild the RadHAR skeleton cache and record its complete pose contract."
    metadata_path = SKRADHAR_ROOT / 'metadata.json'
    signature = {
        'pose_checkpoint': str(checkpoint_path.relative_to(PROJECT_ROOT)),
        'mtime_ns': checkpoint_path.stat().st_mtime_ns,
        'radar_axis_mapping': {'x': 'RadHAR y', 'y': 'RadHAR x', 'z': 'RadHAR z'},
        'radar_frame_alignment': 'translate spatial centroid to MARS training mean',
        'normalized_feature_clip': CONFIG['normalized_feature_clip'],
        'joint_names': list(SKELETON_JOINT_NAMES),
        'coordinates': ['x', 'z'],
        'center_joint': 'SpineShoulder',
    }
    existing = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    if all(existing.get(key) == value for key, value in signature.items()) and not force:
        print(f'Reusing SKradHAR generated with {checkpoint_path.name}')
        return
    shutil.rmtree(SKRADHAR_ROOT / 'Train', ignore_errors=True)
    shutil.rmtree(SKRADHAR_ROOT / 'Test', ignore_errors=True)
    model = load_pose_model(checkpoint_path)
    sources = sorted(RADHAR_ROOT.glob('*/*/*.txt'))
    for index, source_path in enumerate(sources, start=1):
        destination = (SKRADHAR_ROOT / source_path.relative_to(RADHAR_ROOT)).with_suffix('.pt')
        preprocess_radhar_file(source_path, destination, model)
        if index % 20 == 0 or index == len(sources):
            print(f'Preprocessed {index}/{len(sources)} recordings')
    train_files, _ = split_mars_file_paths()
    train_mae, joint_maes = evaluate_pose_checkpoint(checkpoint_path, train_files)
    confidence = 0.85 - 0.1 * torch.log(joint_maes / joint_maes.mean())
    metadata = dict(signature)
    metadata['mars_train_mae_cm'] = train_mae
    metadata['mars_train_joint_mae_cm'] = dict(zip(SKELETON_JOINT_NAMES, joint_maes.tolist()))
    metadata['joint_confidence_scores'] = confidence.tolist()
    metadata['normalized_radhar_feature_summary'] = normalized_radhar_feature_summary()
    SKRADHAR_ROOT.mkdir(parents=True, exist_ok=True)
    metadata_path.write_text(json.dumps(metadata, indent=2))
    print(f'SKradHAR rebuilt with {checkpoint_path.name}')

In [24]:
_, mars_valid_files = split_mars_file_paths()
validation_mae, validation_joint_maes = evaluate_pose_checkpoint(CHECKPOINT_PATH, mars_valid_files)
print(f'MARS validation x,z MAE: {validation_mae:.4f} cm')
print(dict(zip(SKELETON_JOINT_NAMES, validation_joint_maes.tolist())))
print(normalized_radhar_feature_summary())
prepare_skradhar(CHECKPOINT_PATH, force=True)
print(json.loads((SKRADHAR_ROOT / 'metadata.json').read_text()))

MARS validation x,z MAE: 5.2950 cm
{'SpineBase': 2.4003961086273193, 'SpineMid': 0.9982418417930603, 'Neck': 0.576184093952179, 'Head': 1.3631945848464966, 'ShoulderLeft': 4.1050591468811035, 'ElbowLeft': 7.169119834899902, 'WristLeft': 9.811094284057617, 'ShoulderRight': 4.05386209487915, 'ElbowRight': 8.404261589050293, 'WristRight': 11.214695930480957, 'HipLeft': 3.3030529022216797, 'KneeLeft': 5.7454400062561035, 'AnkleLeft': 6.191568374633789, 'FootLeft': 8.52998161315918, 'HipRight': 3.1027772426605225, 'KneeRight': 5.443062782287598, 'AnkleRight': 8.512136459350586, 'FootRight': 9.48079776763916, 'SpineShoulder': 0.20061516761779785}
{'before_clipping': {'x': {'mean': 2.2930979337587587e-09, 'std': 1.0911455154418945}, 'y': {'mean': 1.3069382021058118e-08, 'std': 9.652107238769531}, 'z': {'mean': 2.7695859827758795e-08, 'std': 2.66591739654541}, 'velocity': {'mean': -0.0027827108278870583, 'std': 1.0609211921691895}, 'intensity': {'mean': -0.2866334319114685, 'std': 0.0507667213